In [ ]:
# imports

import os
import sys
import json
import gradio as gr

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv("GOOGLE_API_KEY")

if openai_api_key and openai_api_key.startswith('sk-proj-') and len(openai_api_key)>10:
    print("OpenAI API key looks good so far")
else:
    print("There might be a problem with your OpenAI API key? Please visit the troubleshooting notebook!")
    
if google_api_key and google_api_key.startswith('AI') and len(google_api_key)>10:
    print("Gemini API key looks good so far")
else:
    print("There might be a problem with your Gemini API key? Please visit the troubleshooting notebook!")


openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_relevant_links(url, model, client):
    print(f"Selecting relevant links for {url} by calling {model}")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
def fetch_page_and_all_relevant_links(url, model, client):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url, model, client)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure in Spanish about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url, model, client):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url, model, client)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
def stream_brochure(company_name, url, model, client):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url, model, client)}
          ],
        stream=True
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        # Yield the full response so far for Gradio
        yield response

Here, the selection of AI provider defines the model called. For now, only one model from each of 2 providers has been implemented

In [ ]:
def stream_model(company_name, url, model):
    m = model.strip().lower()
    if m == "gpt":
        yield from stream_brochure(company_name, url, "gpt-4.1-mini", openai)
    elif m == "gemini":
        yield from stream_brochure(company_name, url, "gemini-2.5-flash", gemini)
    else:
        raise ValueError("Unknown model")

UI for the brochure function implemented via GRadio.

In [ ]:
name_input = gr.Textbox(label="Your message:", info="Enter a company name for search ", lines=1)
url_input = gr.Textbox(label="Your message:", info="Enter the company website url ", lines=1)
model_input = gr.Textbox(label="Your message:", info="Enter the llm provider to consult ", lines=1)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="Company Brochure", 
    inputs=[name_input, url_input, model_input], 
    outputs=[message_output], 
    examples=[
        ["Hugginfaces", "https://huggingface.co", "GPT"]
        ], 
    flagging_mode="never"
    )
view.launch()